In [37]:
import torch
from torch.utils.data import DataLoader, TensorDataset, random_split
from collections import OrderedDict
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import scanpy as sc
import numpy as np
import pandas as pd
from torchsummary import summary
from torchviz import make_dot
from tqdm import tqdm
import csv
from scipy.sparse import issparse
from model.sparsemax import Sparsemax

from model.dataset import BagsDataset, custom_collate_fn

In [64]:
adata = sc.read_h5ad('/project/DPDS/Wang_lab/shared/spatial_TCR/data/train_validate/Visium/HumanOvarianCancer/T_cell.h5ad')
dataset = BagsDataset(adata, immune_cell='tcell', radius=150, n_genes=500, resolution='low')
train_size = int(0.7 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
#dataset = load_datasets(data_dir='/project/DPDS/Wang_lab/s439765/spatial_tcr/HumanLungCancerFFPE', immune_cell='tcell',radius=200,)

Tumor cells shape after filtering: (1226, 17943)
Selecting top 500 genes based on mean expression
Percentile value: 4.037618160247803
adata.obs[T] after binarization: AAACAAGTATCTCCCA-1    0
AAACAATCTACTAGCA-1    0
AAACACCAATAACTGC-1    1
AAACAGAGCGACTCCT-1    1
AAACAGCTTTCAGAAG-1    0
Name: T, dtype: int64
Preprocessed data: (3455, 500)


Creating Bags with radius 150: 100%|█████████████████████████| 3455/3455 [00:00<00:00, 23254.69it/s]

Total bags created: 1226
Average instances per bag: 4


In [73]:
dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

distances, gene_expressions, labels, core_index, current_genes = next(iter(dataloader))


In [100]:

class Distance(nn.Module):
    def __init__(self):
        super(Distance, self).__init__()
        self.a = nn.Parameter(torch.tensor(-3.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=0)
        self.softmax = nn.Softmax(dim=0)

    
    def forward(self, x):
        print(x)
        a = self.a
        x = self.softmax(-torch.exp(a) * x)
        return x


In [102]:
model = Distance()
output = model(distances[0])
print(output)

tensor([[  0.0000],
        [141.4214]])
tensor([[9.9913e-01],
        [8.7453e-04]], grad_fn=<SoftmaxBackward0>)


In [103]:

class Gene_expression(nn.Module):
    def __init__(self):
        super(Gene_expression, self).__init__()
        self.b = nn.Parameter(torch.tensor(1.0),requires_grad=True)
        #self.sparsemax = Sparsemax(dim=-1) 
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x):

        b = self.b
        x = self.softmax(torch.exp(b) * x)
        return x


In [104]:
model = Gene_expression()
input_tensor = gene_expressions[0]
output = model(input_tensor)
print(output)

tensor([[6.9253e-01, 9.9309e-02, 1.4810e-02, 7.1933e-02, 2.4732e-02, 3.9375e-02,
         1.0486e-02, 6.1022e-03, 9.3145e-03, 3.8470e-03, 1.8764e-03, 1.1659e-03,
         1.5366e-03, 3.4795e-03, 6.4230e-04, 8.4278e-04, 2.6972e-04, 3.0034e-03,
         4.7494e-04, 2.8409e-04, 1.8186e-04, 5.6023e-04, 2.0475e-04, 8.1405e-04,
         2.0475e-04, 1.9309e-04, 3.4640e-04, 3.8053e-04, 2.2306e-04, 3.1427e-04,
         2.7685e-04, 1.7640e-04, 9.1619e-05, 2.1685e-04, 4.1671e-04, 9.5236e-05,
         1.5561e-04, 6.2924e-05, 1.4108e-04, 1.1883e-04, 3.0655e-04, 1.6580e-04,
         9.5236e-05, 1.2309e-04, 1.3644e-04, 1.2744e-04, 1.2309e-04, 9.1619e-05,
         1.3189e-04, 1.6065e-04, 1.2309e-04, 1.3644e-04, 3.3843e-06, 6.5781e-05,
         9.1619e-05, 9.8941e-05, 1.5561e-04, 1.0662e-04, 9.8941e-05, 9.8941e-05,
         8.4650e-05, 8.8091e-05, 3.6548e-05, 8.1295e-05, 8.8091e-05, 9.1619e-05,
         1.1060e-04, 4.7423e-05, 4.9815e-05, 6.8719e-05, 6.8719e-05, 5.7448e-05,
         4.9815e-05, 2.7399e

In [105]:
df = pd.read_csv('data/tumor_antigens.csv')

In [ ]:

class Immunogenicity(nn.Module):
    def __init__(self, all_genes):
        super(Immunogenicity, self).__init__()
        self.all_genes = all_genes
        self.hla_binder = 
        self.gene_to_index = {gene: idx for idx, gene in enumerate(all_genes)}
        self.hla = 
        self.ig = nn.Parameter(torch.full((len(all_genes),), -1.0), requires_grad=True)
    
    def forward(self, current_genes):
        # Filter genes to include only those present in all_genes
        filtered_genes = [gene for gene in current_genes if gene in self.gene_to_index]
        indices = [self.gene_to_index[gene] for gene in filtered_genes]
        ig = torch.sigmoid(self.ig[indices])
        return ig, filtered_genes


In [ ]:

class MIL(nn.Module):
    def __init__(self, all_genes):
        super(MIL, self).__init__()
        self.distance = Distance()
        self.gene_expression = Gene_expression()
        self.immunogenicity = Immunogenicity(all_genes)
        self.alpha = nn.Parameter(torch.tensor(1.0), requires_grad=True)
        self.beta = nn.Parameter(torch.tensor(1.0), requires_grad=True)
    
    def forward(self, distances, gene_expressions, current_genes):
        bag_outputs = []
        for distance, gene_expression in zip(distances, gene_expressions):
            distance = self.distance(distance)
            gene_expression = self.gene_expression(gene_expression)
            immunogenicity, filtered_genes = self.immunogenicity(current_genes)
            alpha = self.alpha
            beta = self.beta
        
            if len(filtered_genes) == 0:
                continue  # Skip if no overlapping genes
        
        # Print debug information
            #print(f"gene_expression shape: {gene_expression.shape}")
            #print(f"current_genes length: {len(current_genes)}")
            #print(f"filtered_genes length: {len(filtered_genes)}")
        
        
            gene_to_index = {gene: idx for idx, gene in enumerate(current_genes)}
        
            gene_indices = [gene_to_index[gene] for gene in filtered_genes if gene in gene_to_index]
            gene_expression = gene_expression[:, gene_indices]
        
            #print(f"Filtered gene_expression shape: {gene_expression.shape}")
            #print(f"Immunogenicity shape: {immunogenicity.shape}")
        
            z = gene_expression @ immunogenicity
            #print(f"z shape: {z.shape}")
            z = z.unsqueeze(1)
            #print(f"z shape: {z.shape}")
            #print(f"distance shape: {distance}")
            bag_output = distance * z
            bag_output = torch.sum(bag_output, dim=0)
            bag_output = torch.exp(alpha) * bag_output + beta
            bag_output = torch.sigmoid(bag_output)
            #print(bag_output)
            bag_outputs.append(bag_output)
            #df = pd.DataFrame(bag_outputs)
            #df.to_csv('output.csv')
    
        
        return torch.stack(bag_outputs).squeeze(dim=1)
    

class EarlyStopping:
    def __init__(self, patience=5, delta=0.001):
        self.patience = patience
        self.delta = delta
        self.best_loss = np.inf
        self.counter = 0
        self.stopped_epoch = 0
        self.early_stop = False

    def __call__(self, val_loss, model, epoch):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), 'final_model.pth')
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                self.stopped_epoch = epoch